# Example 6.8.1 / 1: A Model-Independent Variable Report

Companion notebook for **6.8.1 Generic Names, Statistics and Message Limits** in the AMPL lab Documentation, section "A Model-Independent Variable Report". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## A Model-Independent Variable Report

The `printf` statement below works for any model. In the Python tab, `ampl` has already read
`model.mod`; `ampl.eval()` runs the indexed `printf`, which has no amplpy method:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var x >= 1, <= 3;
var y {1..2} >= 0;
minimize Total: x + y[1] + y[2];
subject to Cover: x + y[1] >= 2;
subject to Off: y[2] = 0;

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# run.py
ampl.option["solver_msg"] = 0
ampl.solve(solver="highs")
print(ampl.get_value("_nvars"), ampl.get_value("_snvars"))
ampl.eval(r'''
printf {i in 1.._nvars} "%s = %.6g, lb = %.6g, ub = %.6g\n",
    _varname[i], _var[i], _var[i].lb, _var[i].ub;
''')

3 2
x = 1, lb = 1, ub = 3
y[1] = 1, lb = 0, ub = Infinity
y[2] = 0, lb = 0, ub = 0


### Pure AMPL: model.run

The same in pure AMPL, with the same `model.mod`:

In [5]:
%%ampl_eval
reset;
model model.mod;

In [6]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
solve;
display _nvars, _snvars, _ncons, _sncons;
printf {i in 1.._nvars} "%s = %.6g, lb = %.6g, ub = %.6g\n",
    _varname[i], _var[i], _var[i].lb, _var[i].ub;

_nvars = 3
_snvars = 2
_ncons = 2
_sncons = 1

x = 1, lb = 1, ub = 3
y[1] = 1, lb = 0, ub = Infinity
y[2] = 0, lb = 0, ub = 0
